```{=latex}
\usepackage{hyperref}
\usepackage{graphicx}
\usepackage{listings}
\usepackage{textcomp}
\usepackage{fancyvrb}

\newcommand{\passthrough}[1]{\lstset{mathescape=false}#1\lstset{mathescape=false}}
\newcommand{\tightlist}{}
```

```{=latex}
\title{Back Off and Give Up}
\author{Moshe Zadka -- https://cobordism.com}
\date{}

\begin{document}
\begin{titlepage}
\maketitle
\end{titlepage}

\frame{\titlepage}
```

```{=latex}
\begin{frame}
\frametitle{Acknowledgement of Country}

Hayward (in San Francisco Bay Area)

Ancestral homeland of the Ohlone people

\end{frame}
```

I live in Hayward,
in the San Francisco Bay Area.
I wish to acknowledge it as the
ancestral homeland
of the
Ohlone people.

```{=latex}
\begin{frame}
\frametitle{The Secret to Resilient Systems}

\pause

Quit

\end{frame}
```

What if I told you that the secret to building resilient systems is learning when to quit? This might sound counterintuitive - we're taught that persistence is a virtue. But in distributed systems, sometimes giving up is exactly the right strategy. Sometimes, trying harder makes things worse.

```{=latex}
\begin{frame}[fragile]
\frametitle{Naive Retries}

The problem

\end{frame}
```

Part 1: The Problem with Naive Retries

```{=latex}
\begin{frame}[fragile]
\frametitle{Naive Retry: The Classic Mistake}

\begin{verbatim}
import time
import requests

def get_data(url):
    while True:
        try:
            response = requests.get(url, timeout=5)
            return response.json()
        except requests.RequestException:
            time.sleep(1)
\end{verbatim}

\end{frame}
```

Here's code we've all written. Simple retry loop - if it fails, wait a second and try again. Forever. This has infinite retries, fixed delay, no backoff, no jitter, no maximum. When the service recovers, every client hits it simultaneously.

```{=latex}
\begin{frame}
\frametitle{Thundering Herd}

\pause

Recover

\pause

Crash

\end{frame}
```

The thundering herd problem. Service goes down. A thousand clients retry every second. Service recovers after 60 seconds. A thousand simultaneous connections hit it. The service crashes again immediately. This is what happened in our recipe site story.

```{=latex}
\begin{frame}
\frametitle{Retry Storms}

Frontend $\rightarrow$ API Gateway $\rightarrow$ Payment Service

\pause

$3$

\pause

$\times 3$

\pause

$=9$

\pause

$9 \times 100 = 900$

\end{frame}
```

Retry storms. Payment processing service with three layers. Frontend, API Gateway, Payment Service. Each layer has 3 retries. One failed request becomes 27 attempts. At 100 requests per second, that's 2,700 attempts per second. The struggling service gets more load.

```{=latex}
\begin{frame}
\frametitle{Back-off}

Do better!

\end{frame}
```

Part 2: Exponential Back-off with Jitter

```{=latex}
\begin{frame}
\frametitle{Exponential}

Wait

\pause

Proportional to outage

\end{frame}
```

Exponential backoff reduces load quickly as failures continue. Gives the service time to recover. Self-regulating - worse problems get longer waits. Wait time grows with the severity of the problem.

```{=latex}
\begin{frame}[fragile]
\frametitle{Basic Implementation}

\begin{verbatim}
wait_time = 2 ** attempt
time.sleep(wait_time)
\end{verbatim}

\end{frame}
```

Basic exponential backoff implementation. Wait 2 to the power of attempt seconds. Wait times: 1, 2, 4, 8, 16 seconds. But everyone's still synchronized - they all retry at the same times.

```{=latex}
\begin{frame}[fragile]
\frametitle{Jitter}

Without jitter:\pause
Everyone at T=4s

\end{frame}
```

Jitter is critical. Without jitter, all clients retry at exactly 1, 2, 4 seconds. With jitter, we spread them across time windows. Full jitter picks a random time between 0 and the calculated backoff. Instead of everyone at 4 seconds: Client A at 3.2, Client B at 1.8, Client C at 3.9. The load spreads naturally. This randomization prevents the thundering herd.

```{=latex}
\begin{frame}[fragile]
\frametitle{Jitter}
```

In [ ]:
time.sleep(
    random.uniform(0.9, 1.1)
    *
    2**attempt
)

```{=latex}
\end{frame}
```

Jitter is critical. Without jitter, all clients retry at exactly 1, 2, 4 seconds. With jitter, we spread them across time windows. Full jitter picks a random time between 0 and the calculated backoff. Instead of everyone at 4 seconds: Client A at 3.2, Client B at 1.8, Client C at 3.9. The load spreads naturally. This randomization prevents the thundering herd.

```{=latex}
\begin{frame}[fragile]
\frametitle{Maximum timeout}
```

In [ ]:
time.sleep(
    max(
        random.uniform(0.9, 1.1)
        *
        2**attempt,
        MAX_TIMEOUT,
    )
)

```{=latex}
\end{frame}
```

```{=latex}
\begin{frame}[fragile]
\frametitle{What's the maximum?}
```

In [ ]:
MAX_TIMEOUT = timedelta(seconds=30).seconds()
MAX_TIMEOUT = timedelta(minutes=10).seconds()

```{=latex}
\end{frame}
```

```{=latex}
\begin{frame}
\frametitle{A Tale of Two Worlds: Premise}

Recipe recommendation site with ML backend

\end{frame}
```

```{=latex}
\begin{frame}
\frametitle{A Tale of Two Worlds: Premise}

New model rolled out\pause

Overfit on time\pause

Triggers bug

\end{frame}
```

```{=latex}
\begin{frame}
\frametitle{A Tale of Two Worlds: Premise}

Fix\pause

Rolling roll-back

\end{frame}
```

```{=latex}
\begin{frame}
\frametitle{A Tale of Two Worlds: World 1}

Max backoff = 30 second

\end{frame}
```

```{=latex}
\begin{frame}
\frametitle{A Tale of Two Worlds: World 2}

Max backoff = 10 minutes

\end{frame}
```

Let me tell you a story about a recipe recommendation site with an ML backend. During a model rollout, nodes started becoming flaky. In one world - let's call it World 1 - the max backoff was set to 1 second. The front-ends hammered the ML nodes relentlessly, causing a 3-hour complete outage that made the news. But in World 2, where the max backoff was 10 minutes, the system recovered gracefully with just minor degradation that nobody even noticed. One configuration parameter. Two completely different outcomes.

```{=latex}
\begin{frame}[fragile]
\frametitle{Tenacity}

\begin{verbatim}
@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(
        max=600  # 5 minutes!
    )
)
\end{verbatim}

\end{frame}
```

Production code using tenacity. Stop after 5 attempts. Exponential backoff with max of 300 seconds - that's 5 minutes. This is the parameter that saved World 2 in our story.

```{=latex}
\begin{frame}
\frametitle{Giving Up}

Tactical retreat

\end{frame}
```

Part 3: Strategic Giving Up - this is the counterintuitive part.

```{=latex}
\begin{frame}
\frametitle{Why Give Up?}

Avoid catastrophizing

\end{frame}
```

Persistent retrying prevents recovery. Resources tied up in doomed requests. Failures cascade. Strategic abandonment means quick failure, quick recovery. Resources stay available for healthy operations. Partial service beats no service. Sometimes the best thing for a struggling system is to stop using it.

```{=latex}
\begin{frame}
\frametitle{Circuit Breaker}

Three states:

\pause

Closed

\pause

Open

\pause

Half-Open

\end{frame}
```

Circuit breaker pattern - like an electrical circuit breaker. Three states. Closed: normal operation, requests flow through. Open: too many failures, reject immediately without trying. Half-open: test if service recovered with limited traffic. Fails fast when service is down. Automatic recovery detection. Prevents cascade failures.

```{=latex}
\begin{frame}[fragile]
\frametitle{PyBreaker}

\begin{verbatim}
@CircuitBreaker(
    fail_max=5,
    reset_timeout=60
)
def get_user_data(user_id):
    return database.query(...)
\end{verbatim}

\end{frame}
```

PyBreaker implementation. Opens after 5 failures. Tries recovery after 60 seconds. Exclude application errors like KeyError - only trip on infrastructure failures. When open, use a fallback. Degraded service beats no service.

```{=latex}
\begin{frame}
\frametitle{Load Shedding}

Priority levels

\end{frame}
```

Load shedding - choose what to drop. Critical: payments and auth must continue. Important: user updates and searches. Nice-to-have: ML recommendations. Background: reports. At 70% capacity, delay background jobs. 80%, disable recommendations. 90%, simplify search - basic keywords only. 95%, read-only mode. The recipe site could have kept basic search while shedding ML recommendations.

```{=latex}
\begin{frame}[fragile]
\frametitle{Optional Features}

\begin{verbatim}
@optional_feature(fallback=[])
def get_recommendations(user_id):
    return ml_model.predict(user_id)
\end{verbatim}

\end{frame}
```

Decorator for optional features. Above 80% load, return fallback immediately. Aggressive timeout - half second. The recommendations function returns empty list when stressed. Users still get the site.

```{=latex}
\begin{frame}
\frametitle{Putting It Together}

Configuration time

\end{frame}
```

Part 4: Putting It All Together

```{=latex}
\begin{frame}[fragile]
\frametitle{Real Configuration}

\begin{verbatim}
retry:
  max_delay: 300  # 5 minutes!
  jitter: true
    
circuit_breaker:
  failure_threshold: 5
  recovery_timeout: 60
    
load_shedding:
  0.7: ["reports"]
  0.8: ["recommendations"]
  0.9: ["non_critical"]
\end{verbatim}

\end{frame}
```

Real configuration. Max delay: 300 seconds, 5 minutes, human timescale. Jitter enabled. Circuit breaker: 5 failures to open, 60 seconds recovery. Progressive load shedding: 70% drops reports, 80% drops recommendations, 90% drops non-critical. Different timeouts: 5 seconds default, 1 second critical path, 0.5 seconds optional features.

```{=latex}
\begin{frame}
\frametitle{Monitor Everything}

\pause

Metrics

\pause

Alerts

\end{frame}
```

Monitor everything. Track retry success rates - low success means something's wrong. Circuit breaker states - which services struggle. Load shed counts - what functionality are we sacrificing. Alert when breakers stay open too long. Alert when retry success drops below 50%. Alert when load shedding activates. Dashboards need service health maps, retry ratios, load shedding timelines.

##### ```{=latex}
\begin{frame}
\frametitle{Key Takeaways}

\pause

\begin{itemize}
\item Exponential backoff with jitter\pause
\item Max backoff: 5-10 minutes\pause
\item Give up\pause
\item Prioritize
\end{itemize}

\end{frame}
```

Key points. Retries can make things worse - thundering herds, retry storms. Exponential backoff with jitter spreads load. Set max to human timescales: 1 to 10 minutes. Give up strategically - circuit breakers, load shedding. Partial service beats no service. Failing fast means recovering faster.

```{=latex}
\begin{frame}
\frametitle{Resources}

Libraries:
\begin{itemize}
\item \texttt{tenacity}
\item \texttt{pybreaker}
\end{itemize}

\end{frame}
```

Python libraries. Tenacity for comprehensive retries. PyBreaker for circuit breakers.

```{=latex}
\begin{frame}
\frametitle{Questions?}

Moshe Zadka

\url{https://cobordism.com}

\end{frame}
```

Questions? Thank you!

```{=latex}
\end{document}
```